# 01. Exploratory Data Analysis (EDA) — DealSense CRM

In this notebook, we analyze deal volume, stage conversion rates, interaction patterns, response times, and sentiment trends.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

sys.path.append('..')
from training.features import build_features

# Load synthetic deals and labels
data_path = Path('../data/synthetic.json')
labels_path = Path('../data/labels.csv')

with open(data_path, 'r') as f:
    deals = [json.loads(line) for line in f]
labels_df = pd.read_csv(labels_path)

df = build_features(deals, labels_df)
print(f"Total deals loaded: {len(df)}")
df.head()

### 1. Win Rate by Pipeline Stage

In [ ]:
stage_stats = df.groupby('stage').agg(
    deal_count=('deal_id', 'count'),
    win_rate=('is_won', 'mean'),
    avg_value=('deal_value', 'mean')
).reset_index()
print(stage_stats)

plt.figure(figsize=(8, 4))
plt.bar(stage_stats['stage'], stage_stats['win_rate'], color='#3b82f6')
plt.title('Win Rate by Deal Stage')
plt.ylabel('Win Rate')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 2. Interaction Dynamics: Response Times & Stalled Deals (Days Since Last)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

won_deals = df[df['is_won'] == 1]
lost_deals = df[df['is_won'] == 0]

ax1.hist(won_deals['avg_response_min'], bins=20, alpha=0.6, label='Won', color='green')
ax1.hist(lost_deals['avg_response_min'], bins=20, alpha=0.6, label='Lost', color='red')
ax1.set_title('Average Response Time (minutes)')
ax1.legend()

ax2.hist(won_deals['days_since_last'], bins=20, alpha=0.6, label='Won', color='green')
ax2.hist(lost_deals['days_since_last'], bins=20, alpha=0.6, label='Lost', color='red')
ax2.set_title('Days Since Last Interaction')
ax2.legend()

plt.tight_layout()
plt.show()

### 3. Sentiment Trends

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot([won_deals['sentiment_trend'], lost_deals['sentiment_trend']], labels=['Won', 'Lost'])
plt.title('Sentiment Trend (Final - Initial Interaction Sentiment)')
plt.ylabel('Sentiment Delta [-2, 2]')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()